# Validación IndividualFeatureExtractor

Valida el módulo de 5 Individual Features (MDLF, Zhao et al. 2022) para cercanías.

**Antes de ejecutar este notebook necesitas:**
1. `history_df` — historial de validaciones procesadas (TODOS los modos, 1-2 meses).
   Columnas: `id_tarjeta`, `punto_acceso`, `prediccion_descenso`, `timestamp_validacion`, `tipo_transporte`.
2. `x_pred_checked` — salida de `check_transbordo` con columna `tipo` (UNLINKED / BAJADA / TRANSBORDO).
3. `x_test`, `y_test` — salida de `crear_mapeos` (pares subida-descenso REAL).

**Checks del notebook:**
1. Tamaño del training set (UNLINKED ∩ crear_mapeos)
2. Shape de salida
3. Sanity TSI
4. Sanity normalización per-station
5. Sanity PROX: usuario multimodal tiene señal de proximidad
6. Cold-start: usuario sin historial → features cero

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

sys.path.insert(0, '.')
from individual_features import IndividualFeatureExtractor

THETA_CERCANIAS = Path('../LDA/output/theta_cercanias.parquet')
PARADAS_VECINAS = Path('../../topologia/paradas_vecinas_1500m.csv')

# Cargar station_index desde el LDA ya entrenado
theta_cercanias = pd.read_parquet(THETA_CERCANIAS)
station_index = theta_cercanias.index.tolist()
N = len(station_index)
print(f'Estaciones cercanías (N): {N}')
print(f'Primeras 5: {station_index[:5]}')

---
## 0 — Carga de datos de entrada

Adaptar las rutas a los ficheros reales del pipeline.

In [ ]:
# ─── ADAPTAR ESTAS RUTAS ────────────────────────────────────────────
# history_df: validaciones procesadas de 1-2 meses (TODOS los modos)
# Debe tener columnas: id_tarjeta, punto_acceso, prediccion_descenso,
#                      timestamp_validacion, tipo_transporte
# history_df = pd.read_parquet('ruta/al/historial.parquet')

# x_pred_checked: salida de check_transbordo con columna 'tipo'
# x_pred_checked = pd.read_parquet('ruta/a/x_pred_checked.parquet')

# x_test, y_test: salida de crear_mapeos
# x_test = pd.read_parquet('ruta/a/x_test.parquet')
# y_test = pd.read_parquet('ruta/a/y_test.parquet')
# ────────────────────────────────────────────────────────────────────

print('Recuerda asignar: history_df, x_pred_checked, x_test, y_test')

---
## 1 — Tamaño del training set

In [ ]:
# Viajes UNLINKED de cercanías
unlinked_c = x_pred_checked[
    (x_pred_checked['tipo'] == 'UNLINKED') &
    (x_pred_checked['tipo_transporte'] == 'cercanias')
].copy()
print(f'UNLINKED cercanías total:                 {len(unlinked_c):>10,}')

# Intersección con crear_mapeos: UNLINKED que tienen descenso REAL
# x_test tiene las subidas con descenso real asociado en y_test
# Usamos id_tarjeta + timestamp_validacion como clave de join
merge_keys = ['id_tarjeta', 'timestamp_validacion']
train_c = unlinked_c.merge(
    x_test[merge_keys].drop_duplicates(),
    on=merge_keys,
    how='inner',
)
# Añadir etiqueta y (descenso real)
train_c = train_c.merge(
    y_test[merge_keys + ['punto_acceso']].rename(columns={'punto_acceso': 'destino_real'}),
    on=merge_keys,
    how='left',
)

print(f'UNLINKED cercanías con descenso REAL:     {len(train_c):>10,}  ← training set')
print(f'Porcentaje sobre UNLINKED:                {len(train_c)/len(unlinked_c)*100:>9.1f}%')
print()

if len(train_c) < 1000:
    print('⚠ Training set pequeño — considerar incluir BAJADAS normales como datos adicionales')
else:
    print('✓ Tamaño del training set parece suficiente para empezar')

In [ ]:
# Distribución de destinos reales (etiquetas)
top20 = train_c['destino_real'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(top20)), top20.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Top 20 destinos reales en el training set')
ax.set_ylabel('Nº de viajes UNLINKED con ese destino real')
plt.tight_layout()
plt.show()

in_station_index = train_c['destino_real'].isin(station_index).mean() * 100
print(f'Destinos reales que están en station_index: {in_station_index:.1f}%')

---
## 2 — Instanciar y ajustar el extractor

In [ ]:
import time

ext = IndividualFeatureExtractor(
    red='04',
    station_index=station_index,
    proximity_threshold_m=500,
    paradas_vecinas_path=PARADAS_VECINAS,
    norm='l2',
)
print(ext)

t0 = time.time()
ext.fit(history_df)
print(f'fit completado en {time.time()-t0:.1f}s')
print(f'Usuarios en historial cercanías: {len(ext._hist_red):,}')
print(f'Usuarios en historial otros modos: {len(ext._hist_other):,}')
print(f'Paradas en prox_lookup: {len(ext._prox_lookup):,}')

In [ ]:
# Transformar un subconjunto pequeño para validar antes de procesar todo
sample = train_c.head(200).copy()

t0 = time.time()
features_sample = ext.transform(sample)
print(f'transform ({len(sample)} filas) en {time.time()-t0:.1f}s')
print(f'Shape de salida: {features_sample.shape}')
print(f'Esperado:        ({len(sample)}, {N * 8 + 1})')

---
## 3 — Checks de sanidad

In [ ]:
# CHECK 1: Shape
assert features_sample.shape == (len(sample), N * 8 + 1), \
    f'Shape incorrecto: {features_sample.shape}'
print('✓ Shape correcto')

# CHECK 2: has_history ∈ {0, 1}
assert features_sample['has_history'].isin([0.0, 1.0]).all(), \
    'has_history tiene valores distintos de 0/1'
n_with = int(features_sample['has_history'].sum())
n_without = len(features_sample) - n_with
print(f'✓ has_history: {n_with} con historial, {n_without} sin historial (cold-start)')

In [ ]:
# CHECK 3: Cold-start → features todas cero (excepto has_history)
cold_rows = features_sample[features_sample['has_history'] == 0]
if len(cold_rows) > 0:
    feat_cols = [c for c in features_sample.columns if c != 'has_history']
    cold_nonzero = cold_rows[feat_cols].values.sum()
    assert cold_nonzero == 0, f'Cold-start con features no cero: suma={cold_nonzero}'
    print(f'✓ {len(cold_rows)} usuarios cold-start: todas las features son cero')
else:
    print('ℹ No hay usuarios cold-start en la muestra de 200')

In [ ]:
# CHECK 4: Normalización per-station — norma L2 de cada fila de la matriz N×8 ≈ 1
# (salvo las filas donde todos los valores son cero → norma = 0)
feat_cols = [c for c in features_sample.columns if c != 'has_history']
mat = features_sample[feat_cols].values.reshape(len(features_sample), N, 8)
norms = np.linalg.norm(mat, axis=2)  # (n_trips, N)

# Filas con viajes de usuarios CON historial
with_hist = features_sample['has_history'] == 1
if with_hist.any():
    norms_nonzero = norms[with_hist.values].flatten()
    norms_nonzero = norms_nonzero[norms_nonzero > 0]  # excluir estaciones con todo cero
    print(f'Norma L2 media per-station (estaciones no-cero): {norms_nonzero.mean():.4f}')
    assert abs(norms_nonzero.mean() - 1.0) < 0.01, 'Normalización incorrecta'
    print('✓ Normalización per-station correcta')

In [ ]:
# CHECK 5: TSI — sanity manual para un usuario con historial
if with_hist.any():
    # Elegir el primer usuario con historial
    uid_sample = sample.loc[with_hist[with_hist].index[0], 'id_tarjeta']
    hist_u = ext._hist_red.get(uid_sample, pd.DataFrame())
    n_viajes_hist = len(hist_u)
    print(f'Usuario ejemplo: {uid_sample}')
    print(f'  Viajes cercanías en historial: {n_viajes_hist}')

    # TSI debería sumar exactamente n_viajes_hist
    # (solo los destinos que están en station_index)
    tsi_cols = [c for c in features_sample.columns if c.endswith('__tsi')]
    trip_idx = with_hist[with_hist].index[0]
    # Reconstruir valores crudos: la normalización aplana la comparación directa,
    # así que recomputamos con norm='max' para comparar proporcionalmente
    n_dest_in_index = hist_u['destino'].isin(station_index).sum()
    print(f'  Destinos históricos en station_index: {n_dest_in_index}')
    print(f'  (Los destinos fuera de station_index se descartan silenciosamente)')
    print('  ✓ Verificación manual de TSI OK si los números son coherentes')

In [ ]:
# CHECK 6: PROX — usuario multimodal debe tener señal de proximidad
# Buscar un usuario que tenga historial en otros modos Y cuyas paradas estén en el lookup
usuarios_con_other = [uid for uid, g in ext._hist_other.items() if len(g) > 5]
print(f'Usuarios con historial otros modos (>5 validaciones): {len(usuarios_con_other):,}')

# Verificar el lookup: cuántas paradas de otros modos tienen vecina cercanías
print(f'Paradas en prox_lookup: {len(ext._prox_lookup):,}')
if ext._prox_lookup:
    ejemplo_stop, ejemplo_vecinas = next(iter(ext._prox_lookup.items()))
    print(f'Ejemplo: {ejemplo_stop} → vecinas cercanías: {ejemplo_vecinas[:3]}')

# Buscar en la muestra si algún viaje tiene señal de proximidad
prox_tsi_cols = [c for c in features_sample.columns if c.endswith('__prox_tsi')]
prox_sum = features_sample[prox_tsi_cols].sum(axis=1)
n_with_prox = (prox_sum > 0).sum()
print(f'\nViajes con señal PROX (prox_tsi > 0): {n_with_prox} de {len(features_sample)}')
if n_with_prox == 0:
    print('  ℹ Sin señal de proximidad — revisar que paradas_vecinas_1500m.csv se cargó')

---
## 4 — Distribución de señal por feature

In [ ]:
# Para cada sub-feature, ver qué fracción de viajes tienen señal no-nula
sub_feats = ('td_a', 'td_ap', 'sd', 'tsd_b', 'tsd_bp', 'tsi', 'prox_tsi', 'prox_td')
sparsity = {}
for feat in sub_feats:
    cols = [c for c in features_sample.columns if c.endswith(f'__{feat}')]
    nonzero_rows = (features_sample[cols].sum(axis=1) > 0).mean() * 100
    sparsity[feat] = nonzero_rows

fig, ax = plt.subplots(figsize=(10, 3))
bars = ax.bar(sparsity.keys(), sparsity.values(), color='steelblue', alpha=0.8)
ax.set_ylabel('% viajes con señal no-nula')
ax.set_title('Densidad de señal por sub-feature (muestra 200 viajes)')
for bar, val in zip(bars, sparsity.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('\nDensidad por sub-feature:')
for feat, pct in sparsity.items():
    print(f'  {feat:<12} {pct:5.1f}% de viajes con señal')

---
## 5 — Procesar el training set completo y guardar

In [ ]:
import time

t0 = time.time()
features_full = ext.transform(train_c)
print(f'transform completo ({len(train_c):,} viajes) en {time.time()-t0:.1f}s')
print(f'Shape: {features_full.shape}')

# Añadir etiqueta y (destino real) al DataFrame de features
features_full['destino_real'] = train_c['destino_real'].values
features_full['id_tarjeta']   = train_c['id_tarjeta'].values

print(f'\nFracción de destinos reales en station_index: '
      f"{features_full['destino_real'].isin(station_index).mean()*100:.1f}%")

In [ ]:
from pathlib import Path

out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

out_path = out_dir / 'individual_cercanias.parquet'
features_full.to_parquet(out_path)
print(f'Guardado en: {out_path.resolve()}')
print(f'Shape: {features_full.shape}')